# Mini Project 1 (Stars): Start-to-Finish Analysis

This notebook combines the Week 5 data analysis groundwork and the Week 6 visualization work into one place.

## Notebook flow

1. Project overview
2. Data profile
3. Analysis with three charts (one per analytical question)
4. Observations and takeaways


## Project overview

### Dataset
- Source: Kaggle Star Dataset
- Local file used in this repo: `Week 5/stars.csv`
- Core columns: temperature, luminosity, radius, absolute magnitude, star type, star color, spectral class

### Analytical questions (same as MP1a)
1. Which star properties are most useful for telling star types apart?
2. Are larger stars always brighter, or does the relationship change by star type?
3. Does a star's color reliably indicate its temperature and type?


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

STAR_TYPE_LABELS = {
    0: "Brown Dwarf",
    1: "Red Dwarf",
    2: "White Dwarf",
    3: "Main Sequence",
    4: "Supergiant",
    5: "Hypergiant",
}

STAR_TYPE_ORDER = [
    "Brown Dwarf",
    "Red Dwarf",
    "White Dwarf",
    "Main Sequence",
    "Supergiant",
    "Hypergiant",
]

COLOR_NORMALIZATION = {
    "blue white": "Blue-White",
    "blue-white": "Blue-White",
    "bluewhite": "Blue-White",
    "blue": "Blue",
    "red": "Red",
    "white": "White",
    "white-yellow": "White-Yellow",
    "yellow-white": "Yellow-White",
    "yellowish white": "Yellow-White",
    "yellowish": "Yellowish",
    "yellowish-orange": "Yellowish-Orange",
    "orange": "Orange",
    "orange-red": "Orange-Red",
    "pale yellow orange": "Pale Yellow-Orange",
    "whitish": "Whitish",
}


def clean_star_color(value: str) -> str:
    key = str(value).strip().lower().replace("  ", " ").replace("_", " ")
    return COLOR_NORMALIZATION.get(key, str(value).strip().title())


In [ ]:
DATA_PATH = Path("../Week 5/stars.csv")
df = pd.read_csv(DATA_PATH)

df["Star_type_label"] = df["Star type"].map(STAR_TYPE_LABELS)
df["Star_color_clean"] = df["Star color"].apply(clean_star_color)
df["log10_Luminosity"] = df["Luminosity(L/Lo)"].where(df["Luminosity(L/Lo)"] > 0).apply(
    lambda v: np.log10(v) if pd.notna(v) else np.nan
)
df["log10_Radius"] = df["Radius(R/Ro)"].where(df["Radius(R/Ro)"] > 0).apply(
    lambda v: np.log10(v) if pd.notna(v) else np.nan
)

print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())


## Data profile

Before analysis, I profile the data to make sure the table is usable and to understand category balance.


In [ ]:
display(df.head())

print("Star type counts:")
print(df["Star_type_label"].value_counts().reindex(STAR_TYPE_ORDER))

print("\nSpectral class counts:")
print(df["Spectral Class"].value_counts())

print("\nTop cleaned star colors:")
print(df["Star_color_clean"].value_counts().head(10))

## Analysis

This section answers the three MP1a analytical questions with one chart per question.

### Q1. Which star properties are most useful for telling star types apart?

I use a standardized heatmap so different units (K, log luminosity, log radius, magnitude) can be compared on one visual scale.

In [ ]:
metric_cols = [
    "Temperature (K)",
    "log10_Luminosity",
    "log10_Radius",
    "Absolute magnitude(Mv)",
]

means = (
    df.groupby("Star_type_label", observed=True)[metric_cols]
    .mean()
    .reindex(STAR_TYPE_ORDER)
)
zscores = (means - means.mean()) / means.std(ddof=0)
zscores = zscores.rename(
    columns={
        "log10_Luminosity": "Luminosity (L/Lo, log10)",
        "log10_Radius": "Radius (R/Ro, log10)",
        "Absolute magnitude(Mv)": "Absolute Magnitude (Mv)",
    }
)

fig_q1 = go.Figure(
    data=[
        go.Heatmap(
            z=zscores.values,
            x=zscores.columns.tolist(),
            y=zscores.index.tolist(),
            zmid=0,
            colorscale="RdBu",
            reversescale=True,
            text=np.round(zscores.values, 2),
            texttemplate="%{text}",
            colorbar={"title": "Z-score"},
        )
    ]
)
fig_q1.update_layout(
    title="Q1: Star-Type Separation Profile Across Key Properties",
    xaxis_title="Star Property",
    yaxis_title="Star Type",
)
fig_q1.show()

**Observed in Q1:** Luminosity and radius show strong contrast across star types, especially between dwarf classes and giant/hypergiant classes. Temperature helps but is not enough by itself, since white dwarfs can be hot but still low in luminosity/radius.

### Q2. Are larger stars always brighter, or does the relationship change by star type?

I use a log-log scatter because radius and luminosity span large ranges and the question is about relationship between two numeric variables.

In [ ]:
r_raw = df["Radius(R/Ro)"].corr(df["Luminosity(L/Lo)"])
r_log = df["log10_Radius"].corr(df["log10_Luminosity"])
print(f"Pearson r (raw): {r_raw:.3f}")
print(f"Pearson r (log10): {r_log:.3f}")

fig_q2 = px.scatter(
    df,
    x="Radius(R/Ro)",
    y="Luminosity(L/Lo)",
    color="Star_type_label",
    log_x=True,
    log_y=True,
    category_orders={"Star_type_label": STAR_TYPE_ORDER},
    hover_data=["Temperature (K)", "Absolute magnitude(Mv)", "Star_color_clean", "Spectral Class"],
    title="Q2: Radius vs Luminosity by Star Type (Log-Log Scale)",
    labels={
        "Radius(R/Ro)": "Radius (R/Ro, solar radii, log scale)",
        "Luminosity(L/Lo)": "Luminosity (L/Lo, solar luminosities, log scale)",
        "Star_type_label": "Star Type",
    },
)
fig_q2.show()

**Observed in Q2:** The relationship is strongly positive overall (especially in log scale), so larger stars are often brighter. But the clusters by star type show this is not one universal rule; type context matters.

### Q3. Does a star's color reliably indicate its temperature and type?

I use a box plot of temperature by cleaned color categories to test overlap between groups.

In [ ]:
color_order = (
    df.groupby("Star_color_clean", observed=True)["Temperature (K)"]
    .mean()
    .sort_values()
    .index.tolist()
)

fig_q3 = px.box(
    df,
    x="Star_color_clean",
    y="Temperature (K)",
    color="Star_type_label",
    category_orders={"Star_color_clean": color_order, "Star_type_label": STAR_TYPE_ORDER},
    title="Q3: Temperature Distribution by Star Color and Star Type",
    labels={
        "Star_color_clean": "Star Color (cleaned categories)",
        "Temperature (K)": "Temperature (K)",
        "Star_type_label": "Star Type",
    },
)
fig_q3.show()

display(
    df.groupby("Star_color_clean", observed=True)["Temperature (K)"]
    .agg(["count", "mean", "min", "max"])
    .sort_values("mean")
    .round(1)
)

**Observed in Q3:** Red groups are generally cooler while blue and blue-white groups are hotter, but there is overlap. So color is a useful clue, not a standalone classifier for star type.

## Final observations

- The three charts answer three different MP1a questions using chart types that match each data relationship.
- Luminosity and radius are strong separators of star type in this dataset.
- Radius and luminosity are strongly related overall, but star type-specific patterns still matter.
- Color trends with temperature directionally, but overlap means it should be combined with other properties.

This notebook can be used as the start-to-finish analysis narrative for MP1b.